# SDTM Instrument Identity Enrich — Attach NCIt Identity to Instrument Codelists

Reads the instrument test-code extract from `SDTM_CT_Extract` and enriches
each instrument codelist with NCIt identity from two independent NCIt branches:

- **C20993** (Research or Clinical Assessment Tool) — the instrument concept itself
- **C211913** (CDISC QRS Instruments Questions) — the question container that groups the codelist's TESTCDs

These two branches are **not** structurally linked in NCIt. `C211913 → codelist` is a
structural 1:1 mapping via NCIt parent-child walks. `C20993 → codelist` is a name-based
mapping (exact / suffix-strip / normalized) that tops out around 70%. Token-overlap
fuzzy matching is deliberately excluded — it produced wrong-version matches
(HAMD-17→HAMD-21, SF-36→SF-8, PHQ-8→PHQ-4) in exploratory work.

A third NCIt anchor, the codelist concept itself (CDISC Terminology branch, e.g. C67153
for "ADAS-Cog CDISC Version Test Code"), is carried through from the Extract step as
`Codelist_NCIt_Code`. It is a dead end for structural linking to either C20993 or C211913.

## Pipeline Position

```
sdtm-test-codes/
├── notebooks/
│   ├── SDTM_CT_Extract.ipynb                  ← upstream (produces instrument extract CSV)
│   ├── SDTM_CT_NCIt_Enrich.ipynb              ← sibling (test-level enrichment)
│   └── SDTM_Instrument_Identity_Enrich.ipynb  ← this notebook (codelist-level)
├── downloads/
│   ├── Thesaurus.txt                          ← NCIt FLAT
│   └── nci_code_cui_map.dat                   ← NCIt→UMLS
├── interim/
│   ├── SDTM_CT_Instrument_Extract.csv         ← upstream input
│   ├── C20993_Instrument_Hierarchy.csv        ← produced here
│   └── C211913_Question_Containers.csv        ← produced here
└── machine_actionable/
    └── SDTM_Instrument_Identity.xlsx          ← produced here (one row per codelist)
```

## Design Decisions

- **Two NCIt anchors, explicitly prefixed** — `Instrument_NCIt_*` (C20993, identity) and
  `Container_NCIt_*` (C211913, structure). The structural gap between the two branches
  is a property of NCIt itself; explicit column separation names it rather than hiding it.
- **Token-overlap matching excluded** — name similarity is not the same as instrument
  identity. Codelists without a clean name match get `Instrument_Match_Method = UNMATCHED`
  and empty `Instrument_NCIt_*` columns, not a silent fuzzy guess.
- **Codelist grain** — one row per codelist. Sibling to the test-within-codelist grain in
  `SDTM_Instrument_Test_Identity.xlsx`, joinable via `Codelist_TESTCD`.


## 1. Configuration


In [ ]:
import re
import csv
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

BASE_DIR      = Path.cwd().parent
DOWNLOADS_DIR = BASE_DIR / 'downloads'
INTERIM_DIR   = BASE_DIR / 'interim'
MA_DIR        = BASE_DIR / 'machine_actionable'

MA_DIR.mkdir(exist_ok=True)

# Inputs
INPUT_CSV     = INTERIM_DIR / 'SDTM_CT_Instrument_Extract.csv'
FLAT_FILE     = DOWNLOADS_DIR / 'Thesaurus.txt'
CUI_MAP_FILE  = DOWNLOADS_DIR / 'nci_code_cui_map.dat'

# Outputs
OUTPUT_XLSX         = MA_DIR / 'SDTM_Instrument_Identity.xlsx'
C20993_INTERIM_CSV  = INTERIM_DIR / 'C20993_Instrument_Hierarchy.csv'
C211913_INTERIM_CSV = INTERIM_DIR / 'C211913_Question_Containers.csv'

# Anchor NCIt concepts
C20993_CODE  = 'C20993'   # Research or Clinical Assessment Tool (instrument identity)
C211913_CODE = 'C211913'  # CDISC QRS Instruments Questions (question containers)

for f in [INPUT_CSV, FLAT_FILE, CUI_MAP_FILE]:
    if not f.exists():
        raise FileNotFoundError(f'Input not found: {f}')

print(f'Input CSV:    {INPUT_CSV}')
print(f'FLAT file:    {FLAT_FILE}')
print(f'CUI map:      {CUI_MAP_FILE}')
print(f'Output xlsx:  {OUTPUT_XLSX}')


## 2. Load CT Instrument Extract


In [ ]:
df_ext = pd.read_csv(INPUT_CSV, dtype=str).fillna('')

required_cols = ['NCIt_code', 'TESTCD', 'domain', 'instrument_label',
                 'codelist_testcd', 'codelist_ncit_code']
missing = [c for c in required_cols if c not in df_ext.columns]
if missing:
    raise ValueError(f'Required columns missing from CT extract: {missing}')

print(f'Rows:               {len(df_ext):,}')
print(f'Distinct codelists: {df_ext["codelist_testcd"].nunique()}')
print(f'Domains:            {sorted(df_ext["domain"].unique())}')


## 3. Parse NCIt FLAT (with parent relationships) + CUI map

FLAT tab-separated columns:
`code | uri | parent_code | names(pipe-separated) | definition | display_name | concept_status | semantic_type | codelist_membership`

First name in the pipe-separated list is the preferred term; the rest are synonyms.
CUI map is pipe-separated `ncit_code | cui_value`. UMLS CUIs match `C\d+`, NCI Metathesaurus CUIs match `CL\d+`.


In [ ]:
parent_map = {}   # code -> parent code
name_map   = {}   # code -> preferred term
syn_map    = {}   # code -> [synonyms]
defn_map   = {}   # code -> definition

with open(FLAT_FILE, 'r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        if len(row) < 4:
            continue
        code = row[0]
        parent_map[code] = row[2]
        names = row[3].split('|')
        name_map[code] = names[0].strip()
        syn_map[code]  = [n.strip() for n in names[1:] if n.strip()]
        defn_map[code] = row[4].strip() if len(row) > 4 else ''

children_of = defaultdict(set)
for code, parent in parent_map.items():
    if parent:
        children_of[parent].add(code)

print(f'NCIt concepts loaded: {len(parent_map):,}')
print(f'Parent-child links:   {sum(len(v) for v in children_of.values()):,}')


In [ ]:
umls_cui_map = {}
ncim_cui_map = {}

with open(CUI_MAP_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            ncit_code = parts[0].strip()
            cui_val   = parts[1].strip()
            if re.match(r'^C\d+$', cui_val):
                umls_cui_map[ncit_code] = cui_val
            elif re.match(r'^CL\d+$', cui_val):
                ncim_cui_map[ncit_code] = cui_val

print(f'UMLS CUI entries: {len(umls_cui_map):,}')
print(f'NCIm CUI entries: {len(ncim_cui_map):,}')


## 4. C20993 — Research or Clinical Assessment Tool descendants

Transitive descendant set. Persisted as interim CSV with depth, parent, and
preferred terms for inspection.


In [ ]:
def get_descendants(root, children_index):
    out = set()
    queue = list(children_index.get(root, set()))
    while queue:
        c = queue.pop()
        if c not in out:
            out.add(c)
            queue.extend(children_index.get(c, set()))
    return out

def depth_from(code, root, parent_index, max_depth=20):
    d = 0
    c = code
    while c != root and d < max_depth:
        c = parent_index.get(c, '')
        d += 1
    return d if c == root else -1

c20993_all = get_descendants(C20993_CODE, children_of)

c20993_rows = []
for code in c20993_all:
    c20993_rows.append({
        'NCIt_Code':      code,
        'Preferred_Term': name_map.get(code, ''),
        'Parent_Code':    parent_map.get(code, ''),
        'Parent_Term':    name_map.get(parent_map.get(code, ''), ''),
        'Depth':          depth_from(code, C20993_CODE, parent_map),
        'Synonyms':       ' | '.join(syn_map.get(code, [])),
        'Definition':     defn_map.get(code, ''),
        'UMLS_CUI':       umls_cui_map.get(code, ''),
        'NCIm_CUI':       ncim_cui_map.get(code, ''),
    })

c20993_df = pd.DataFrame(c20993_rows).sort_values(['Depth', 'Preferred_Term']).reset_index(drop=True)
c20993_df.to_csv(C20993_INTERIM_CSV, index=False)

print(f'C20993 descendants: {len(c20993_df):,}')
print(f'Saved: {C20993_INTERIM_CSV}')
print(f'Depth distribution:')
for d, n in c20993_df['Depth'].value_counts().sort_index().items():
    print(f'  Depth {d}: {n:,}')


## 5. C211913 — CDISC QRS Instruments Questions (question containers)

Direct children of C211913. Each is a container grouping the NCIt question concepts
that correspond to SDTM CT TESTCDs in one codelist.


In [ ]:
c211913_children = children_of.get(C211913_CODE, set())

# Count descendants per container (depth > 1 children — the questions themselves)
container_descendant_counts = {
    sub: len(get_descendants(sub, children_of)) for sub in c211913_children
}

c211913_rows = []
for code in c211913_children:
    c211913_rows.append({
        'NCIt_Code':        code,
        'Preferred_Term':   name_map.get(code, ''),
        'Descendant_Count': container_descendant_counts[code],
        'Synonyms':         ' | '.join(syn_map.get(code, [])),
        'Definition':       defn_map.get(code, ''),
        'UMLS_CUI':         umls_cui_map.get(code, ''),
        'NCIm_CUI':         ncim_cui_map.get(code, ''),
    })

c211913_df = pd.DataFrame(c211913_rows).sort_values('Preferred_Term').reset_index(drop=True)
c211913_df.to_csv(C211913_INTERIM_CSV, index=False)

print(f'C211913 direct children (question containers): {len(c211913_df)}')
print(f'Saved: {C211913_INTERIM_CSV}')
print(f'Descendant count range: {c211913_df["Descendant_Count"].min()} to {c211913_df["Descendant_Count"].max()}')


## 6. Build codelist-grain base from CT extract

Group member rows by `codelist_testcd`. Each grouping attribute (`instrument_label`,
`domain`, `codelist_ncit_code`) is uniform across members — validate before aggregating.
`Question_Count` is the number of rows with a populated `TESTCD` (i.e. codes present
in the Test Code codelist itself, excluding rows that only carry the paired Test Name).


In [ ]:
# Validate per-codelist uniformity of grouping attributes
for col in ['instrument_label', 'domain', 'codelist_ncit_code']:
    dup = df_ext.groupby('codelist_testcd')[col].nunique()
    violators = dup[dup > 1]
    if len(violators):
        raise ValueError(f'Codelist has multiple {col} values: {violators.to_dict()}')

base = df_ext.groupby('codelist_testcd').agg(
    Instrument_Label   = ('instrument_label',    'first'),
    Domain             = ('domain',              'first'),
    Codelist_NCIt_Code = ('codelist_ncit_code',  'first'),
    Question_Count     = ('TESTCD', lambda x: (x != '').sum()),
).reset_index().rename(columns={'codelist_testcd': 'Codelist_TESTCD'})

print(f'Codelists: {len(base)}')
print(f'Domain breakdown: {base["Domain"].value_counts().to_dict()}')
print(f'Total TESTCDs across all codelists: {base["Question_Count"].sum():,}')
print(f'Question count range: {base["Question_Count"].min()} to {base["Question_Count"].max()}')


## 7. Structural mapping: C211913 → codelist (Container_NCIt_Code)

For each C211913 child, walk its NCIt descendants and intersect with the NCIt codes of
TESTCDs in the extract. Descendants that belong to exactly one codelist give an
unambiguous `Container_NCIt_Code` for that codelist. Containers that span multiple
codelists (1:many) are skipped.


In [ ]:
# NCIt code -> codelist submission value (green side)
ncit_to_codelist = dict(zip(df_ext['NCIt_code'], df_ext['codelist_testcd']))
green_ncit_codes = set(ncit_to_codelist)

# For each C211913 container, what codelists do its NCIt descendants fall into?
container_to_codelists = {}  # container_code -> set of codelist_testcd
for sub in c211913_children:
    desc = get_descendants(sub, children_of)
    hits = desc & green_ncit_codes
    if hits:
        container_to_codelists[sub] = {ncit_to_codelist[h] for h in hits}

# Assign container to codelist only when 1:1
codelist_to_container = {}
for container, cls in container_to_codelists.items():
    if len(cls) == 1:
        codelist_to_container[next(iter(cls))] = container

one_to_one  = sum(1 for cls in container_to_codelists.values() if len(cls) == 1)
one_to_many = sum(1 for cls in container_to_codelists.values() if len(cls) > 1)

print(f'C211913 children:                {len(c211913_children)}')
print(f'Containers overlapping green:    {len(container_to_codelists)}')
print(f'  1:1 (assigned to codelist):    {one_to_one}')
print(f'  1:many (ambiguous, skipped):   {one_to_many}')
print(f'Codelists with container assigned: {len(codelist_to_container)} of {len(base)}')

base['Container_NCIt_Code'] = base['Codelist_TESTCD'].map(codelist_to_container).fillna('')
n_cont = (base['Container_NCIt_Code'] != '').sum()
print(f'Container coverage: {n_cont} / {len(base)} = {100*n_cont/len(base):.1f}%')


## 8. Identity mapping: codelist → C20993 (Instrument_NCIt_Code)

Cascade of name-based matching strategies, exclusively non-fuzzy. Each strategy is
applied only to codelists unmatched by the previous one.

1. **exact** — `Instrument_Label` matches a C20993 preferred term or synonym, case-insensitive.
2. **suffix-strip** — strip trailing type suffix ("Questionnaire", "Scale", "Functional Test", …).
3. **normalized** — lowercase, hyphen normalization, version marker removal, whitespace collapse.
4. **normalized-item-strip** — `normalized` + remove "- N Item" / "N Items" item-count qualifiers.

Token-overlap / Jaccard fuzzy matching is excluded — it produced wrong-version matches
in exploratory work. Rows without a clean name match get
`Instrument_Match_Method = UNMATCHED` and empty `Instrument_NCIt_*` columns.


In [ ]:
TYPE_SUFFIXES = [
    ' functional test', ' questionnaire', ' clinical classification',
    ' clinical score', ' score', ' scale', ' grade',
    ' clinical category', ' subscale', ' memory aid',
]

def strip_type_suffix(label):
    lower = label.lower()
    for suffix in TYPE_SUFFIXES:
        if lower.endswith(suffix):
            return label[:len(label) - len(suffix)].strip()
    return label

def normalize_name(name):
    n = name.lower()
    if n.startswith('the '):
        n = n[4:]
    n = re.sub(r'\s*[-–—]\s*', '-', n)
    n = n.replace(' test', ' functional test')
    n = re.sub(r'\s+v\d+\.?\d*', '', n)
    n = re.sub(r'\s+version\s+\S+', '', n)
    n = re.sub(r'\s+\d{1,2}[a-z]{3}\d{4}', '', n)
    n = re.sub(r'\s+', ' ', n).strip()
    return n

# Build indices over C20993 descendants (preferred term + synonyms)
c20993_name_index = {}  # lower(name) -> code
c20993_norm_index = {}  # normalize_name(name) -> code

for code in c20993_all:
    pref = name_map.get(code, '')
    if pref:
        c20993_name_index.setdefault(pref.lower(), code)
        c20993_norm_index.setdefault(normalize_name(pref), code)
    for syn in syn_map.get(code, []):
        if syn:
            c20993_name_index.setdefault(syn.lower(), code)
            c20993_norm_index.setdefault(normalize_name(syn), code)

print(f'C20993 name index:       {len(c20993_name_index):,} entries')
print(f'C20993 normalized index: {len(c20993_norm_index):,} entries')

# Apply cascade
codes_out   = []
methods_out = []

for _, row in base.iterrows():
    label = row['Instrument_Label']

    # 1. exact
    if label.lower() in c20993_name_index:
        codes_out.append(c20993_name_index[label.lower()])
        methods_out.append('exact')
        continue

    # 2. suffix-strip
    stripped_lower = strip_type_suffix(label).lower()
    if stripped_lower in c20993_name_index:
        codes_out.append(c20993_name_index[stripped_lower])
        methods_out.append('suffix-strip')
        continue

    # 3. normalized (on suffix-stripped)
    norm_label = normalize_name(strip_type_suffix(label))
    if norm_label in c20993_norm_index:
        codes_out.append(c20993_norm_index[norm_label])
        methods_out.append('normalized')
        continue

    # 4. normalized-item-strip
    norm2 = re.sub(r'-\s*\d+\s*items?', '', norm_label).strip()
    norm2 = re.sub(r'\s+\d+\s*items?', '', norm2).strip()
    if norm2 != norm_label and norm2 in c20993_norm_index:
        codes_out.append(c20993_norm_index[norm2])
        methods_out.append('normalized-item-strip')
        continue

    codes_out.append('')
    methods_out.append('UNMATCHED')

base['Instrument_NCIt_Code']    = codes_out
base['Instrument_Match_Method'] = methods_out

print()
print('Match method breakdown:')
for method, count in base['Instrument_Match_Method'].value_counts().items():
    print(f'  {method:25s} {count:4d}')


## 9. Enrich with Instrument_NCIt_* and Container_NCIt_* columns

Attach preferred term, synonyms (pipe-separated, undifferentiated), definition,
UMLS CUI, and NCIm CUI for both anchors where the NCIt code is populated. Unpopulated
codes produce empty strings — no placeholder values.


In [ ]:
def lookup(series, attr_map):
    return series.apply(lambda c: attr_map.get(c, '') if c else '')

def lookup_syn(series):
    return series.apply(lambda c: ' | '.join(syn_map.get(c, [])) if c else '')

# Instrument anchor (C20993)
base['Instrument_NCIt_Preferred_Term'] = lookup(base['Instrument_NCIt_Code'], name_map)
base['Instrument_NCIt_Synonyms']       = lookup_syn(base['Instrument_NCIt_Code'])
base['Instrument_NCIt_Definition']     = lookup(base['Instrument_NCIt_Code'], defn_map)
base['Instrument_UMLS_CUI']            = lookup(base['Instrument_NCIt_Code'], umls_cui_map)
base['Instrument_NCIm_CUI']            = lookup(base['Instrument_NCIt_Code'], ncim_cui_map)

# Container anchor (C211913)
base['Container_NCIt_Preferred_Term']  = lookup(base['Container_NCIt_Code'], name_map)
base['Container_NCIt_Synonyms']        = lookup_syn(base['Container_NCIt_Code'])
base['Container_NCIt_Definition']      = lookup(base['Container_NCIt_Code'], defn_map)
base['Container_UMLS_CUI']             = lookup(base['Container_NCIt_Code'], umls_cui_map)
base['Container_NCIm_CUI']             = lookup(base['Container_NCIt_Code'], ncim_cui_map)

print(f'Instrument preferred term populated: {(base["Instrument_NCIt_Preferred_Term"] != "").sum()} / {len(base)}')
print(f'Container  preferred term populated: {(base["Container_NCIt_Preferred_Term"] != "").sum()} / {len(base)}')
print(f'Instrument UMLS CUI populated:       {(base["Instrument_UMLS_CUI"] != "").sum()} / {len(base)}')
print(f'Container  UMLS CUI populated:       {(base["Container_UMLS_CUI"] != "").sum()} / {len(base)}')


## 10. Coverage report


In [ ]:
has_inst = base['Instrument_NCIt_Code'] != ''
has_cont = base['Container_NCIt_Code'] != ''

both       = int((has_inst & has_cont).sum())
inst_only  = int((has_inst & ~has_cont).sum())
cont_only  = int((~has_inst & has_cont).sum())
neither    = int((~has_inst & ~has_cont).sum())
total      = len(base)

print('Coverage matrix:')
print(f'  Both Instrument + Container NCIt: {both:4d} ({100*both/total:.1f}%)')
print(f'  Instrument only (no container):   {inst_only:4d} ({100*inst_only/total:.1f}%)')
print(f'  Container only (no instrument):   {cont_only:4d} ({100*cont_only/total:.1f}%)')
print(f'  Neither:                          {neither:4d} ({100*neither/total:.1f}%)')

print()
print('By Domain:')
for dom, g in base.groupby('Domain'):
    n_i = (g['Instrument_NCIt_Code'] != '').sum()
    n_c = (g['Container_NCIt_Code']  != '').sum()
    print(f'  {dom}: {len(g):4d} codelists  |  Instrument match: {n_i:4d}  |  Container: {n_c:4d}')

print()
print('Instrument match method × Domain:')
xtab = pd.crosstab(base['Instrument_Match_Method'], base['Domain'], margins=True, margins_name='Total')
print(xtab.to_string())


## 11. Write SDTM_Instrument_Identity.xlsx

Two sheets: **README** (provenance, scope, columns, coverage, design decisions, known
limitations) and **Instruments** (one row per codelist).


In [ ]:
COLUMN_ORDER = [
    # Neutral (keys + base identity)
    'Codelist_TESTCD',
    'Codelist_NCIt_Code',
    'Instrument_Label',
    'Domain',
    'Question_Count',
    # Instrument anchor (C20993)
    'Instrument_NCIt_Code',
    'Instrument_NCIt_Preferred_Term',
    'Instrument_NCIt_Synonyms',
    'Instrument_NCIt_Definition',
    'Instrument_UMLS_CUI',
    'Instrument_NCIm_CUI',
    'Instrument_Match_Method',
    # Container anchor (C211913)
    'Container_NCIt_Code',
    'Container_NCIt_Preferred_Term',
    'Container_NCIt_Synonyms',
    'Container_NCIt_Definition',
    'Container_UMLS_CUI',
    'Container_NCIm_CUI',
]

NEUTRAL_COLS = {
    'Codelist_TESTCD', 'Codelist_NCIt_Code',
    'Instrument_Label', 'Domain', 'Question_Count',
}
INSTRUMENT_COLS = {
    'Instrument_NCIt_Code', 'Instrument_NCIt_Preferred_Term',
    'Instrument_NCIt_Synonyms', 'Instrument_NCIt_Definition',
    'Instrument_UMLS_CUI', 'Instrument_NCIm_CUI',
    'Instrument_Match_Method',
}
CONTAINER_COLS = {
    'Container_NCIt_Code', 'Container_NCIt_Preferred_Term',
    'Container_NCIt_Synonyms', 'Container_NCIt_Definition',
    'Container_UMLS_CUI', 'Container_NCIm_CUI',
}

COLUMN_WIDTHS = {
    'Codelist_TESTCD': 15,
    'Codelist_NCIt_Code': 14,
    'Instrument_Label': 55,
    'Domain': 7,
    'Question_Count': 11,
    'Instrument_NCIt_Code': 14,
    'Instrument_NCIt_Preferred_Term': 45,
    'Instrument_NCIt_Synonyms': 55,
    'Instrument_NCIt_Definition': 60,
    'Instrument_UMLS_CUI': 14,
    'Instrument_NCIm_CUI': 14,
    'Instrument_Match_Method': 22,
    'Container_NCIt_Code': 14,
    'Container_NCIt_Preferred_Term': 45,
    'Container_NCIt_Synonyms': 55,
    'Container_NCIt_Definition': 60,
    'Container_UMLS_CUI': 14,
    'Container_NCIm_CUI': 14,
}

out_df = base[COLUMN_ORDER].sort_values(['Domain', 'Codelist_TESTCD']).reset_index(drop=True)

# ---- Three-layer color palette ----
# Instrument level (C20993 â parent)   â dark chocolate + warm cream
INSTRUMENT_HEADER = PatternFill('solid', fgColor='5B3A29')
INSTRUMENT_LIGHT  = PatternFill('solid', fgColor='EDE0D4')
# Container level (C211913 â middle)   â copper + light peach
CONTAINER_HEADER  = PatternFill('solid', fgColor='CC7A3E')
CONTAINER_LIGHT   = PatternFill('solid', fgColor='FAEADC')
# Neutral (codelist keys + base)       â grey
NEUTRAL_HEADER    = PatternFill('solid', fgColor='7F7F7F')
NEUTRAL_LIGHT     = PatternFill('solid', fgColor='F2F2F2')
# README section headers               â warm tan
BROWN_SECTION     = PatternFill('solid', fgColor='D4A574')

WHITE_BOLD   = Font(name='Arial', bold=True, size=10, color='FFFFFF')
TITLE_FONT   = Font(name='Arial', bold=True, size=14, color='4A2F1A')
SECTION_FONT = Font(name='Arial', bold=True, size=11, color='4A2F1A')
BODY_FONT    = Font(name='Arial', size=10)
BOLD_FONT    = Font(name='Arial', bold=True, size=10)
THIN_BORDER  = Border(bottom=Side(style='thin', color='D9D9D9'))

def col_fills(col_name):
    if col_name in INSTRUMENT_COLS:
        return INSTRUMENT_HEADER, INSTRUMENT_LIGHT
    if col_name in CONTAINER_COLS:
        return CONTAINER_HEADER, CONTAINER_LIGHT
    return NEUTRAL_HEADER, NEUTRAL_LIGHT

# ---- Coverage counters for README ----
n_total   = len(out_df)
n_inst    = int((out_df['Instrument_NCIt_Code'] != '').sum())
n_cont    = int((out_df['Container_NCIt_Code']  != '').sum())
n_both    = int(((out_df['Instrument_NCIt_Code'] != '') & (out_df['Container_NCIt_Code'] != '')).sum())
n_neither = int(((out_df['Instrument_NCIt_Code'] == '') & (out_df['Container_NCIt_Code'] == '')).sum())
n_umls_i  = int((out_df['Instrument_UMLS_CUI'] != '').sum())
n_umls_c  = int((out_df['Container_UMLS_CUI']  != '').sum())

method_counts = out_df['Instrument_Match_Method'].value_counts().to_dict()
def mcount(k): return method_counts.get(k, 0)

wb = Workbook()

# ---- README sheet ----
ws_readme = wb.active
ws_readme.title = 'README'
ws_readme.sheet_properties.tabColor = '5B3A29'

def add_row(ws, r, text, font=BODY_FONT, fill=None):
    c = ws.cell(row=r, column=1, value=text if text else None)
    c.font = font
    c.alignment = Alignment(wrap_text=True, vertical='top')
    if fill:
        c.fill = fill
    return r + 1

r = 1
r = add_row(ws_readme, r, 'SDTM INSTRUMENT IDENTITY — NCIt ENRICHED REFERENCE FILE', TITLE_FONT)
r += 1

r = add_row(ws_readme, r, 'PROVENANCE', SECTION_FONT, BROWN_SECTION)
r = add_row(ws_readme, r, f'Generated:      {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
r = add_row(ws_readme, r, 'Pipeline:       SDTM_CT_Extract → SDTM_Instrument_Identity_Enrich')
r = add_row(ws_readme, r, 'SDTM CT source: NCI EVS SDTM Terminology (versionless URL, re-run to update)')
r = add_row(ws_readme, r, 'NCIt source:    Thesaurus.txt from Thesaurus.FLAT.zip (NCI EVS)')
r = add_row(ws_readme, r, 'CUI source:     nci_code_cui_map.dat (NCI EVS, NCIm release schedule)')
r += 1

r = add_row(ws_readme, r, 'SCOPE', SECTION_FONT, BROWN_SECTION)
r = add_row(ws_readme, r, 'Instrument codelists from SDTM CT non-extensible Test Code codelists.')
r = add_row(ws_readme, r, f'One row per codelist. {n_total} codelists across QS / FT / RS.')
r = add_row(ws_readme, r, 'Sibling file at test-within-codelist grain: SDTM_Instrument_Test_Identity.xlsx')
r = add_row(ws_readme, r, 'Join key between the two files: Codelist_TESTCD')
r += 1

r = add_row(ws_readme, r, 'COLUMN DESCRIPTIONS', SECTION_FONT, BROWN_SECTION)
r = add_row(ws_readme, r, 'Codelist_TESTCD                 Primary key. Submission value of the Test Code codelist (e.g. ADCTC)')
r = add_row(ws_readme, r, 'Codelist_NCIt_Code              NCIt concept code for the codelist itself (CDISC Terminology branch)')
r = add_row(ws_readme, r, 'Instrument_Label                Display name minus " Test Code" suffix')
r = add_row(ws_readme, r, 'Domain                          QS / FT / RS (or ?? for TAUG-origin codelists outside SDTMIG v3.4)')
r = add_row(ws_readme, r, 'Question_Count                  Number of TESTCDs in the codelist')
r = add_row(ws_readme, r, '')
r = add_row(ws_readme, r, 'Instrument_NCIt_Code            NCIt code for the instrument-as-tool (C20993 branch)')
r = add_row(ws_readme, r, 'Instrument_NCIt_Preferred_Term  Preferred term from NCIt FLAT')
r = add_row(ws_readme, r, 'Instrument_NCIt_Synonyms        Pipe-separated synonyms (undifferentiated) from NCIt FLAT')
r = add_row(ws_readme, r, 'Instrument_NCIt_Definition      Plain-language definition from NCIt FLAT')
r = add_row(ws_readme, r, 'Instrument_UMLS_CUI             Standard UMLS CUI (C + digits)')
r = add_row(ws_readme, r, 'Instrument_NCIm_CUI             NCI Metathesaurus CUI (CL + digits)')
r = add_row(ws_readme, r, 'Instrument_Match_Method         Resolution path: exact / suffix-strip / normalized / normalized-item-strip / UNMATCHED')
r = add_row(ws_readme, r, '')
r = add_row(ws_readme, r, 'Container_NCIt_Code             NCIt code for the question container (C211913 branch)')
r = add_row(ws_readme, r, 'Container_NCIt_Preferred_Term   Preferred term from NCIt FLAT')
r = add_row(ws_readme, r, 'Container_NCIt_Synonyms         Pipe-separated synonyms from NCIt FLAT')
r = add_row(ws_readme, r, 'Container_NCIt_Definition       Plain-language definition from NCIt FLAT')
r = add_row(ws_readme, r, 'Container_UMLS_CUI              Standard UMLS CUI')
r = add_row(ws_readme, r, 'Container_NCIm_CUI              NCI Metathesaurus CUI')
r += 1

r = add_row(ws_readme, r, 'COVERAGE STATISTICS', SECTION_FONT, BROWN_SECTION)
r = add_row(ws_readme, r, f'Total codelists:                 {n_total}', BOLD_FONT)
r = add_row(ws_readme, r, f'Instrument NCIt populated:       {n_inst} ({100*n_inst/n_total:.1f}%)')
r = add_row(ws_readme, r, f'Container  NCIt populated:       {n_cont} ({100*n_cont/n_total:.1f}%)')
r = add_row(ws_readme, r, f'Both populated:                  {n_both} ({100*n_both/n_total:.1f}%)')
r = add_row(ws_readme, r, f'Neither populated:               {n_neither} ({100*n_neither/n_total:.1f}%)')
r = add_row(ws_readme, r, f'Instrument UMLS CUI populated:   {n_umls_i} ({100*n_umls_i/n_total:.1f}%)')
r = add_row(ws_readme, r, f'Container  UMLS CUI populated:   {n_umls_c} ({100*n_umls_c/n_total:.1f}%)')
r = add_row(ws_readme, r, '')
r = add_row(ws_readme, r, 'Instrument match method:', BOLD_FONT)
r = add_row(ws_readme, r, f'  exact:                 {mcount("exact"):4d}')
r = add_row(ws_readme, r, f'  suffix-strip:          {mcount("suffix-strip"):4d}')
r = add_row(ws_readme, r, f'  normalized:            {mcount("normalized"):4d}')
r = add_row(ws_readme, r, f'  normalized-item-strip: {mcount("normalized-item-strip"):4d}')
r = add_row(ws_readme, r, f'  UNMATCHED:             {mcount("UNMATCHED"):4d}')
r += 1

r = add_row(ws_readme, r, 'DESIGN DECISIONS', SECTION_FONT, BROWN_SECTION)
r = add_row(ws_readme, r, 'Two NCIt anchors, explicitly prefixed. C20993 (Research or Clinical Assessment')
r = add_row(ws_readme, r, 'Tool) carries instrument identity. C211913 (CDISC QRS Instruments Questions) carries')
r = add_row(ws_readme, r, 'question-container structure. The two branches are not linked in NCIt — explicit')
r = add_row(ws_readme, r, 'column separation names the gap rather than hiding it.')
r = add_row(ws_readme, r, '')
r = add_row(ws_readme, r, 'Token-overlap fuzzy matching excluded. Exploratory work found wrong-version errors')
r = add_row(ws_readme, r, '(HAMD-17 → HAMD-21, SF-36 → SF-8). Rows without a clean exact / suffix /')
r = add_row(ws_readme, r, 'normalized match get UNMATCHED with empty Instrument_NCIt_* columns.')
r = add_row(ws_readme, r, '')
r = add_row(ws_readme, r, 'Codelist grain. One row per codelist. Test-within-codelist grain lives in')
r = add_row(ws_readme, r, 'SDTM_Instrument_Test_Identity.xlsx, joinable via Codelist_TESTCD.')
r += 1

r = add_row(ws_readme, r, 'COLOR CONVENTION', SECTION_FONT, BROWN_SECTION)
r = add_row(ws_readme, r, 'Dark chocolate (#5B3A29)  — Instrument identity columns (C20993, parent level)')
r = add_row(ws_readme, r, 'Copper (#CC7A3E)          — Container / subscale columns (C211913, middle level)')
r = add_row(ws_readme, r, 'Grey (#7F7F7F)            — codelist key columns + base identity')
r = add_row(ws_readme, r, 'Green (#548235)           — TESTCD identity (sibling Test_Identity files)')
r = add_row(ws_readme, r, 'Yellow (#FFD700)          — COSMoS Dataset Specialization layer (consumer files)')
r += 1

r = add_row(ws_readme, r, 'KNOWN LIMITATIONS', SECTION_FONT, BROWN_SECTION)
r = add_row(ws_readme, r, '1. Instrument identity (C20993) is name-based — fragile for versioned variants,')
r = add_row(ws_readme, r, '   respondent forms, and disease-specific modules. See Instrument_Match_Method.')
r = add_row(ws_readme, r, '2. C211913 containers spanning multiple codelists (1:many) are excluded.')
r = add_row(ws_readme, r, '3. Codelists with Domain == ?? are TAUG-origin (outside SDTMIG v3.4 scope).')

ws_readme.column_dimensions['A'].width = 100

# ---- Instruments data sheet ----
ws_data = wb.create_sheet('Instruments')
ws_data.sheet_properties.tabColor = '5B3A29'

# Pre-compute per-column fills (header + alt-row) once
col_header_fills = {}
col_alt_fills    = {}
for ci, col in enumerate(COLUMN_ORDER, start=1):
    h, a = col_fills(col)
    col_header_fills[ci] = h
    col_alt_fills[ci]    = a

# Header row
for ci, col in enumerate(COLUMN_ORDER, start=1):
    cell = ws_data.cell(row=1, column=ci, value=col)
    cell.font = WHITE_BOLD
    cell.fill = col_header_fills[ci]
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = THIN_BORDER
    ws_data.column_dimensions[get_column_letter(ci)].width = COLUMN_WIDTHS.get(col, 18)
ws_data.row_dimensions[1].height = 28

# Data rows — alternating fill per column group
for ri, row in enumerate(out_df.itertuples(index=False), start=2):
    is_alt = (ri % 2 == 0)
    for ci, col in enumerate(COLUMN_ORDER, start=1):
        val = getattr(row, col)
        cell = ws_data.cell(row=ri, column=ci, value=str(val) if val else '')
        cell.font = BODY_FONT
        cell.alignment = Alignment(wrap_text=True, vertical='top')
        cell.border = THIN_BORDER
        if is_alt:
            cell.fill = col_alt_fills[ci]

ws_data.freeze_panes = 'A2'
ws_data.auto_filter.ref = f'A1:{get_column_letter(len(COLUMN_ORDER))}{ws_data.max_row}'

wb.save(OUTPUT_XLSX)
print(f'Saved: {OUTPUT_XLSX}')
print(f'  Rows:   {len(out_df)}')
print(f'  Sheets: {wb.sheetnames}')
